# Week 7 — Cross-Validation

## Trusting Your Score

In the previous tasks, I compared different machine learning models and improved the input features.

The engineered K-Nearest Neighbors model achieved a high accuracy using one train/test split. However, one split may give a lucky or unlucky result.

In this notebook, I will use 5-fold cross-validation to check whether the model performs consistently across different parts of the dataset.

## Project Question

**Is the accuracy of my engineered KNN model reliable across different parts of the Wine dataset?**

I will compare the earlier single train/test split accuracy with the average accuracy from 5-fold cross-validation.

## Weakness of a Single Train/Test Split

A single train/test split divides the dataset only once.

The final accuracy may change depending on which rows are selected for training and testing. The model may receive an easier test set and achieve a high score, or it may receive a harder test set and achieve a lower score.

Because of this, one split may not provide a complete picture of the model's true performance.

## What Is K-Fold Cross-Validation?

K-fold cross-validation divides the dataset into a selected number of sections called folds.

For 5-fold cross-validation, the model is trained and tested five times. During each round, four folds are used for training and one fold is used for testing.

Each fold becomes the test set once.

After all five rounds, the accuracy scores are averaged to produce a more reliable result.

## Simple 5-Fold Diagram

| Round | Training Folds | Testing Fold |
|---|---|---|
| 1 | 2, 3, 4, 5 | 1 |
| 2 | 1, 3, 4, 5 | 2 |
| 3 | 1, 2, 4, 5 | 3 |
| 4 | 1, 2, 3, 5 | 4 |
| 5 | 1, 2, 3, 4 | 5 |

The model is tested on every part of the dataset once.

## Why the Average Score Is More Trustworthy

The cross-validation average is based on several different test groups instead of only one.

This reduces the effect of receiving one lucky or unlucky test set.

The standard deviation also shows whether the model performs consistently. A small standard deviation means that the fold scores are reasonably close to each other.

## Overfitting

Overfitting happens when a model performs very well on training data but performs poorly on unseen test data.

This can happen when the model learns the training examples too closely instead of learning general patterns.

Cross-validation can help detect this problem because the model is tested on several different groups of unseen data.

In [2]:
# Import the required libraries

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:
# Load the Wine dataset

wine = load_wine()

print("The Wine dataset has been loaded successfully.")

The Wine dataset has been loaded successfully.


In [4]:
# Convert the dataset into a pandas DataFrame

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

# Add the target column

df["target"] = wine.target

df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [5]:
# Check the dataset size and target classes

print("Dataset shape:", df.shape)

print("\nNumber of samples in each class:")
print(df["target"].value_counts().sort_index())

Dataset shape: (178, 14)

Number of samples in each class:
target
0    59
1    71
2    48
Name: count, dtype: int64


## Dataset

The Wine dataset contains 178 wine samples.

It has 13 original numerical features and three target classes.

The target represents the correct wine class that the model will try to predict.

In [6]:
# Create a categorical feature from the alcohol column

df["alcohol_level"] = pd.cut(
    df["alcohol"],
    bins=[-np.inf, 12.5, 13.5, np.inf],
    labels=["Low", "Medium", "High"]
)

df[["alcohol", "alcohol_level"]].head()

,alcohol,alcohol_level
0,14.23,High
1,13.20,Medium
2,13.16,Medium
3,14.37,High
4,13.24,Medium


In [7]:
# Create the phenol ratio feature

df["phenol_ratio"] = (
    df["total_phenols"] /
    (df["flavanoids"] + 0.001)
)

df[
    [
        "total_phenols",
        "flavanoids",
        "phenol_ratio"
    ]
].head()

,total_phenols,flavanoids,phenol_ratio
0,2.80,3.06,0.914734
1,2.65,2.76,0.959797
2,2.80,3.24,0.863931
3,3.85,3.49,1.102836
4,2.80,2.69,1.040505


## Feature Engineering Used

I continued with the engineered features from the previous task.

The alcohol_level feature groups the alcohol measurement into Low, Medium and High categories.

The phenol_ratio feature represents the relationship between total phenols and flavanoids.

The numerical columns will also be scaled because KNN calculates distance and can be affected by features with different value ranges.

In [8]:
# Select the input features

original_numeric_features = list(wine.feature_names)

numeric_features = (
    original_numeric_features +
    ["phenol_ratio"]
)

categorical_features = [
    "alcohol_level"
]

feature_columns = (
    numeric_features +
    categorical_features
)

X = df[feature_columns]
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (178, 15)
y shape: (178,)


## Understanding X and y

X contains the information used by the model.

It includes the original chemical measurements, the new phenol ratio and the alcohol level category.

y contains the correct wine class that the model must predict.

In [9]:
# Prepare the numerical and categorical features

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

## Why Preprocessing Is Inside a Pipeline

The numerical scaler should learn only from the training data.

During cross-validation, every fold has a different training section. Placing StandardScaler inside the pipeline makes sure that scaling is learned separately during each fold.

This prevents information from the test fold from entering the training process.

In [10]:
# Create the engineered KNN pipeline

knn_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=5
            )
        )
    ]
)

print("The KNN pipeline has been created.")

The KNN pipeline has been created.


In [11]:
# Create the same single train/test split used earlier

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 142
Testing samples: 36


In [12]:
# Train the pipeline using one train/test split

knn_pipeline.fit(
    X_train,
    y_train
)

# Make predictions

single_split_predictions = knn_pipeline.predict(
    X_test
)

# Calculate the single-split accuracy

single_split_accuracy = accuracy_score(
    y_test,
    single_split_predictions
)

print(
    f"Single-Split Accuracy: "
    f"{single_split_accuracy:.2%}"
)

Single-Split Accuracy: 100.00%


## Single-Split Observation

The engineered KNN model achieved approximately 100% accuracy using this single train/test split.

This is a strong result, but it is based on only one testing group. Cross-validation will show whether the model performs similarly on other parts of the dataset.

In [13]:
# Create 5 balanced folds

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold cross-validation has been prepared.")

5-fold cross-validation has been prepared.


## Why StratifiedKFold Is Used

This is a classification problem with three target classes.

StratifiedKFold tries to keep a similar proportion of each wine class inside every fold.

This makes the folds more balanced and the evaluation fairer.

In [14]:
# Run 5-fold cross-validation

cv_scores = cross_val_score(
    knn_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy"
)

print("Cross-validation completed.")

Cross-validation completed.


In [15]:
# Print every fold score

for fold_number, score in enumerate(
    cv_scores,
    start=1
):
    print(
        f"Fold {fold_number} Accuracy: "
        f"{score:.2%}"
    )

Fold 1 Accuracy: 97.22%
Fold 2 Accuracy: 91.67%
Fold 3 Accuracy: 100.00%
Fold 4 Accuracy: 97.14%
Fold 5 Accuracy: 100.00%


In [16]:
# Calculate the mean and standard deviation

cv_mean = cv_scores.mean()
cv_standard_deviation = cv_scores.std()

print(
    f"Cross-Validation Mean Accuracy: "
    f"{cv_mean:.2%}"
)

print(
    f"Cross-Validation Standard Deviation: "
    f"{cv_standard_deviation:.2%}"
)

print(
    f"\nFinal CV Result: "
    f"{cv_mean:.2%} ± "
    f"{cv_standard_deviation:.2%}"
)

Cross-Validation Mean Accuracy: 97.21%
Cross-Validation Standard Deviation: 3.04%

Final CV Result: 97.21% ± 3.04%


## Understanding Mean and Standard Deviation

The mean accuracy represents the average result across all five folds.

The standard deviation shows how much the accuracy changed between the folds.

A cross-validation result of approximately 97.21% ± 3.04% means that the model performed very well overall, but the exact accuracy changed slightly depending on which data was used for testing.

In [17]:
# Create a comparison table

comparison = pd.DataFrame({
    "Evaluation Method": [
        "Single Train/Test Split",
        "5-Fold Cross-Validation Average"
    ],
    "Accuracy": [
        single_split_accuracy,
        cv_mean
    ]
})

comparison["Accuracy (%)"] = (
    comparison["Accuracy"] * 100
).round(2)

comparison[
    [
        "Evaluation Method",
        "Accuracy (%)"
    ]
]

,Evaluation Method,Accuracy (%)
0,Single Train/Test Split,100.00
1,5-Fold Cross-Validation Average,97.21


In [18]:
# Calculate the difference between the scores

score_difference = (
    single_split_accuracy -
    cv_mean
) * 100

print(
    f"Difference between the scores: "
    f"{score_difference:.2f} percentage points"
)

Difference between the scores: 2.79 percentage points


## Comparison With the Earlier Score

The earlier single train/test split accuracy was approximately 100.00%.

The 5-fold cross-validation average was approximately 97.21%, with a standard deviation of approximately 3.04%.

The cross-validation average was around 2.79 percentage points lower than the single-split accuracy.

This suggests that the original test set may have been slightly easier for the model. The model still performed very well, but cross-validation gave a more realistic view of its performance across different parts of the dataset.

## Which Score Do I Trust More, and Why?

I trust the 5-fold cross-validation average more than the single train/test split accuracy.

The single-split score was based on only one group of test examples, so the model may have received a particularly easy test set.

Cross-validation tested the model five times using different sections of the dataset. Every section was used for testing once, and the final score was calculated from the average of all five results.

For this reason, the cross-validation average gives me a more stable and realistic estimate of how the model may perform on unseen data.

## Overfitting Interpretation

The cross-validation average remained high and the fold scores were reasonably close to each other.

This does not show a major overfitting problem in this comparison.

However, the fact that the single-split accuracy was higher than the cross-validation average shows why one perfect score should not be trusted without checking the model on different data splits.

## Final Conclusion

In this task, I used 5-fold cross-validation to evaluate my engineered KNN model more reliably.

The model achieved a cross-validation average of approximately 97.21% with a standard deviation of approximately 3.04%.

This was slightly lower than the earlier single-split accuracy of 100.00%, but it provided a more trustworthy result because the model was trained and tested on five different data sections.

This task helped me understand that a high score from one split does not always represent the model's true performance.